# U0.L4 — U-Net Skeleton and Eval Harness

**Flow-Based Generative Models · UFRJ · 2026.2** · lab notebook · date: see calendar-map

Today you write the two pieces of code the rest of the course imports without
modification, and you **freeze** both interfaces at the end.

- **`unet-skeleton`** — U3.L2 trains it, U3.L3 conditions it, PS0 imports it.
- **`eval-harness`** — every number this course reports from here on.

**How this lab works.** Numbered **TODO** cells are yours; the primitive blocks
(ResBlock with FiLM, attention, up/down-sampling) are **given complete**, and you are
asked to read them — there is a checkpoint question about one of them. **Checkpoint
cells** appear every ~15 minutes: *if the assert passes, continue.*

**Format note.** This lab's deliverable is a **package**, not a notebook. The code you
write lands in `common/unet.py` and `common/eval.py`, beside the `hygiene-stack` from
U0.L2, because that is where importable code belongs. This notebook is the driver: the
checks, the pictures, and nothing that another session needs to import.

Full prose, every figure and every number: the [U0.L4 lecture notes](../docs/U0/U0-L4-notes.html).

## Part A — The contracts

*Reused in: U3.L2, U3.L3, U1.L2, PS1.4, PS0 — that is, everywhere.*

Every previous lab **ended** by freezing an interface. This one **starts** by freezing
two, because the freeze is the deliverable. Read both before writing anything.

```
UNet(in_ch, base_ch, ch_mults, attn_resolutions, num_classes=None)
forward(x: (B,C,H,W) float, t: (B,) float in [0,1], y: (B,) long opt)
    -> (B,C,H,W)
```

1. **`t` is always `(B,)` floats in $[0,1]$** — the FM arrow at code level, asserted in
   `forward`. Any code needing another time variable converts *outside* the network.
   This one line is what keeps the diffusion dialect out of the codebase.
2. **`y` is optional**, and the label embedding is *summed* into the time embedding.
3. **The output has the input's shape.**

```
common/eval.py
  FeatureExtractor;  MNISTFeatures (pinned);  InceptionV3Features
  ref_stats(loader, extractor, dataset=, split=) -> RefStats
  fid(samples, ref, extractor) -> float
  bpd(log_prob_nats_per_example, num_dims) -> float   # PER-EXAMPLE nats
  panel(sample_fn, eps_bank, path) -> figure
```

**The comparability rule:** an FID is a property of a model *and* a measurement
protocol — extractor, preprocessing, sample count. The harness enforces it.

In [ ]:
import math
import random
import sys
from pathlib import Path

# The notebook lives in labs/; `common/` sits at the repository root.
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    """Seed all three RNG streams (python, numpy, torch incl. CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(0)

### Read the provided blocks

Three of them, all short. You are asked about `num_groups` at the Part B checkpoint.

Run the next cell and read the source it prints. The two things to notice: the
modulation lands **after** the normalization (that is the whole point of FiLM — the
norm has just removed the activation's scale and offset, and modulation puts a
conditioning-dependent one back), and several layers are **zero-initialized**.

In [ ]:
import inspect

from common.unet import AttnBlock, ResBlock, num_groups

print(inspect.getsource(num_groups))
print(inspect.getsource(ResBlock.forward))

## Part B — Assemble the U-Net

*Reused in: every time-conditioned model in the course.*

The build map is U0.T4's anatomy diagram — keep it on screen.

### TODO 1 — the sinusoidal ladder

The spec is frozen at U0.T4 and you copy it exactly. For `t` a `(B,)` float tensor in
$[0,1]$ and an even width `dim`, with `half = dim // 2`:

$$\omega_k = (10^4)^{-k/\mathrm{half}}, \qquad
  \mathrm{emb}(t) = [\,\sin(\omega_k \cdot 1000\,t)\;;\;
                       \cos(\omega_k \cdot 1000\,t)\,]_k$$

The factor $1000$ is the **only** rescale, and it is deliberate: it makes this
from-scratch ladder produce the *same numbers* as the DiT and ADM reference
implementations, so a later cross-check against a library reports a real bug rather
than a convention difference. Outside this function, `t` is a float in $[0,1]$
everywhere.

In [ ]:
def timestep_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """(B,) float in [0,1] -> (B, dim). TODO 1."""
    if dim % 2 != 0:
        raise ValueError(f"embedding width must be even, got {dim}")
    half = dim // 2
    # TODO 1a: geometric frequencies omega_k, k = 0 .. half-1.
    #          Hint: exp(-log(10000) * arange(half) / half)
    freqs = ...
    # TODO 1b: the argument. Remember the factor 1000, and that t is (B,)
    #          so the result must be (B, half).
    arg = ...
    # TODO 1c: sine and cosine, concatenated on the last axis.
    return ...

**Checkpoint 1.1** — the ladder against the frozen reference. This is the cross-check
the factor $1000$ exists for: if it fails, you have a convention difference, and a
convention difference here becomes a silent numerical difference against every library
you compare with later.

In [ ]:
from common.unet import timestep_embedding as reference_ladder

t_probe = torch.tensor([0.0, 0.25, 0.5, 1.0])
mine = timestep_embedding(t_probe, 128)
assert mine.shape == (4, 128), mine.shape
assert torch.allclose(mine, reference_ladder(t_probe, 128), atol=1e-6)
# The slowest rung is nearly linear in t; the fastest completes ~159 cycles.
print("ladder OK  |  emb(0) norm:", float(mine[0].norm()))

### TODO 2 — wire the architecture

This is the honest difficulty of the session, and it is **bookkeeping**, not insight.
The encoder pushes an activation onto a stack at every step; the decoder pops one and
**concatenates** it on the channel axis before its residual block runs. The decoder's
input width therefore depends on what the encoder pushed.

Reference configuration: `in_ch=1`, `base_ch=64`, `ch_mults=(1, 2, 2)`,
`attn_resolutions=(7,)`.

| Level | Resolution | Channels | Attention |
|---|---|---|---|
| 0 | 28 × 28 | 64 | — |
| 1 | 14 × 14 | 128 | — |
| 2 | 7 × 7 | 128 | yes |
| bottleneck | 7 × 7 | 128 | yes |

**The one-extra-block rule.** The decoder gets one *more* residual block per level than
the encoder. The level's last skip is the one the downsampling layer pushed, and it
needs a block of its own to be consumed. Matching block counts leaves one skip on the
stack, and the failure is a shape error deep inside the last level rather than anything
legible.

### TODO 3 — the conditioning path

$$c = \mathrm{emb}(t) + \mathrm{emb}(y) \;\longrightarrow\;
     \text{MLP} \;\longrightarrow\; (\gamma, \beta)$$

The label table has the **ladder's** width, not `d_emb`, because the sum happens
*before* the projection. One projection, whatever the number of conditioning signals.

Write both TODOs into `common/unet.py` (the file carries the surrounding structure and
the provided blocks), then import the result below.

**Checkpoint 2.1 — Part B.** Three things pass before you continue.

In [ ]:
from common.unet import UNet

model = UNet(in_ch=1, base_ch=64, ch_mults=(1, 2, 2), attn_resolutions=(7,))
model_cond = UNet(1, 64, (1, 2, 2), (7,), num_classes=10)

# 1. Shapes, at two resolutions.
x = torch.randn(4, 1, 28, 28)
t = torch.rand(4)
# TODO: assert the output shape at 28x28, then again on a 14x14 input.
...

# 2. Parameter count within 5% of the reference.
n = sum(p.numel() for p in model.parameters())
# TODO: assert n is within 5% of 6_946_881, and print the conditional count.
...

# 3. The reading question. What does num_groups return for 64, 128 and 1,
#    and why is the answer not always 32? Answer in the cell below.
print(model.describe())

## Part C — Is the conditioning alive?

*Reused in: every time-conditioned model henceforth.*

A wired network that runs is not a working network. Three cheap checks, under a minute
together, each of which has caught a real bug in this course's own code.

**They cannot all run at the same moment, and that is the lesson of this part.** The
zero-initialization makes the untrained network **exactly** the zero function, for every
$t$. So:

- **Ritual 1** (the zero-field check) can only run **at initialization**;
- **Rituals 2 and 3** (is $t$ wired? is $y$ wired?) read the modulation **inside** the
  network, where the zero output head cannot reach, so they run at initialization too;
- **Ritual 4** (output sensitivity) measures at the output and therefore moves the
  parameters first — one plain SGD step, on a copy that is then discarded.

The older reading, in which sensitivity is measured at the output at step 0, is **vacuous** — a network
  whose output is identically zero cannot tell two times apart.

This is the second time the course meets this shape; U0.L3's gradient check had the same
defect. The rule: *a check that asserts a network does something cannot run at an
initialization designed to make it do nothing.*

In [ ]:
# Ritual 1 FIRST, at initialization. Exact equality against zero is
# deliberate here: 'approximately zero' would hide a lost zero-init.
fresh = UNet(1, 64, (1, 2, 2), (7,), num_classes=10).to(DEVICE)
x1 = torch.randn(4, 1, 28, 28, device=DEVICE)
with torch.no_grad():
    out0 = fresh(x1, torch.zeros(4, device=DEVICE))
    out1 = fresh(x1, torch.ones(4, device=DEVICE))
assert float(out0.abs().max()) == 0.0, "a zero-init was lost"
assert float(out1.abs().max()) == 0.0, "a zero-init was lost"
print("ritual 1 OK — the untrained field is the zero field")

### Moving the parameters is not training

Thirty optimizer steps against **random targets**: no dataset, no generative objective,
result discarded. **The U-Net is not trained today** — its first real training is U3.L2,
by design. This only leaves the zero function so the other rituals have something to
measure.

In [ ]:
def wake_up(model, steps: int = 30, seed: int = 0) -> None:
    """Leave the zero function. A diagnostic, NOT a training run."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for _ in range(steps):
        x = torch.randn(8, 1, 28, 28, generator=g).to(DEVICE)
        t = torch.rand(8, generator=g).to(DEVICE)
        target = torch.randn(8, 1, 28, 28, generator=g).to(DEVICE)
        loss = ((model(x, t) - target) ** 2).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    model.eval()


wake_up(fresh)

**Checkpoint 3.1 — the sensitivity rituals.** They catch the single most common silent
failure of a time-conditioned model: a conditioning path wired to nothing. It still
trains, the loss still falls, and the model is a $t$-independent average of what it
should have learned.

In [ ]:
ts = torch.linspace(0.0, 1.0, 41)
with torch.no_grad():
    norms = [float(fresh(x1, torch.full((4,), float(v), device=DEVICE)).norm())
             for v in ts]

# TODO: assert the spread over t is strictly positive, and
#                  report it relative to the mean.
...

# TODO (ritual 3): at fixed (x, t), assert the outputs at ten different
#                  labels are pairwise distinct.
...

plt.plot(ts.numpy(), norms)
plt.xlabel("$t$"); plt.ylabel(r"$\|f(x,t)\|$"); plt.show()

### The gradient-check ritual

Finite differences against autograd, in the U0.L1 and U0.L3 lineage, on one FiLM
projection weight — the parameter through which the conditioning signal reaches the
computation at all.

**Run it in float64.** In float32 a central difference over a sum of some fifteen
hundred activations loses every digit it needs; the first version of this check in the
course's own build reported a relative error of 155, which was the arithmetic failing
and not the gradient.

And **self-test it**: re-run the same comparison against a deliberately doubled
gradient, where it must fail. A check that has never failed is not known to work.

In [ ]:
fresh = fresh.double()
xg = torch.randn(2, 1, 28, 28, generator=torch.Generator().manual_seed(3))
xg, tg = xg.double().to(DEVICE), torch.rand(2).double().to(DEVICE)
film = next(m.film for m in fresh.modules() if hasattr(m, "film"))

fresh.zero_grad(set_to_none=True)
fresh(xg, tg).sum().backward()
analytic = film.weight.grad.detach().clone()
assert float(analytic.abs().max()) > 0, "vacuous — wake_up first"

idx = np.unravel_index(int(analytic.abs().argmax()), tuple(analytic.shape))
eps = 1e-6
with torch.no_grad():
    w0 = film.weight[idx].item()
    film.weight[idx] = w0 + eps
    plus = float(fresh(xg, tg).sum())
    film.weight[idx] = w0 - eps
    minus = float(fresh(xg, tg).sum())
    film.weight[idx] = w0
numeric, a = (plus - minus) / (2 * eps), float(analytic[idx])

rel = abs(numeric - a) / abs(a)
self_test = abs(numeric - 2 * a) / abs(2 * a)   # must FAIL, i.e. be large
assert rel < 1e-2, rel
assert self_test > 0.1, "the check cannot fail — it measures nothing"
fresh.float()
print(f"gradient check OK — rel error {rel:.2e}; "
      f"self-test (must be large) {self_test:.2f}")

## Part D — The evaluation harness

*Reused in: U1.L2 and PS1.4 (`bpd`), U3.L2 and U3.L3 (`fid`, `ref_stats`, `panel`),
PS0 (all).*

U0.T4 wrote the definitions. Here they become code — and two of the three caveats
become properties of the program rather than things to remember.

### TODO 4 — `ref_stats`, and why float64

Push a loader through the extractor; return the mean and covariance of the features;
cache to disk, because the reference side does not change between models.

**The accumulation runs in float64, and this is not fastidiousness.** A covariance sums
products across orders of magnitude; in float32 the small terms are lost to rounding
against a growing sum, and the matrix can fail to be positive semi-definite. The failure
then surfaces two functions away, inside a matrix square root, as a complex number
nobody expected.

**The cache key IS caveat (i), written as a filename:**

```
refstats__mnist-test__mnist-cnn-v1__01-float-28x28.npz
```

A key without the preprocessing serves the wrong pipeline on a cache hit — silently,
permanently, and exactly as the caveat warns.

In [ ]:
def ref_stats_body(loader, extractor):
    """TODO 4: running mean and covariance, in float64."""
    total, s1, s2 = 0, None, None
    for batch in loader:
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        # TODO 4a: features, cast to float64 immediately.
        f = ...
        if s1 is None:
            s1 = np.zeros(f.shape[1], dtype=np.float64)
            s2 = np.zeros((f.shape[1], f.shape[1]), dtype=np.float64)
        # TODO 4b: accumulate the sum and the sum of outer products.
        ...
        total += f.shape[0]
    # TODO 4c: mean, then the UNBIASED covariance from the running sums.
    mu = ...
    sigma = ...
    return mu, sigma, total

### TODO 5 — the Fréchet distance, and the wrong fix

$$\mathrm{FID} = \lVert \mu_r - \mu_g \rVert^2
  + \operatorname{Tr}\!\big( \Sigma_r + \Sigma_g
    - 2 (\Sigma_r \Sigma_g)^{1/2} \big)$$

`scipy.linalg.sqrtm` returns a **complex** array. On a product of two covariances the
true root is real, but floating-point error decorates it with a tiny imaginary part.

The standard fix — call `.real` — is wrong, and only *sometimes*. A **large** imaginary
part means the input was not positive semi-definite: a genuine upstream bug, usually a
float32 covariance. So truncate, but measure first.

In [ ]:
from scipy.linalg import sqrtm


def sqrtm_psd(a, tol: float = 1e-3):
    """TODO 5a: matrix square root with the imaginary-part guard."""
    root = sqrtm(a)
    if np.iscomplexobj(root):
        # TODO: measure the imaginary part relative to the real scale.
        #       Raise if it exceeds tol; only then take .real.
        ...
    return np.asarray(root, dtype=np.float64)


def fid_body(feats, mu_r, sigma_r):
    """TODO 5b: the formula above, on float64 features."""
    ...

### Caveat (i), made unavoidable

`fid` returns a `FIDValue`. It **is** a `float` — it prints, compares and sorts as one —
and it additionally carries the extractor, the preprocessing and the sample count, and
refuses a comparison across a different protocol.

That machinery is worth its cost. A docstring warning does not survive contact with a
results table, where two numbers sit in one column with nothing to say how each was
measured. U0.T4 measured what happens then: score a genuinely better model with fewer
samples than its rival and it loses in every trial.

### TODO 6 — bits per dimension

$$\mathrm{BPD} = -\frac{\mathbb{E}_{x \sim q}[\log p_\theta(x)]}{d \log 2}$$

Mechanical today, with the argument in nats and $d = 784$ for MNIST.
**The dequantization term is deliberately absent** — it is added when U1 first *uses*
this number, and the docstring carries the marker so that its absence is never mistaken
for a decision nobody made.

### TODO 7 — the panel

Formalize U0.L3's trick: a **fixed** noise bank, drawn once and reused at every
evaluation, so two panels differ because the model changed and not because the noise
did. Without it, every real change smaller than the sampling noise is invisible.

In [ ]:
def bpd_body(mean_log_lik_nats: float, num_dims: int) -> float:
    """TODO 6."""
    ...


# TODO 7: write `panel` into common/eval.py, then check it here on the
#         identity 'model': a fixed bank of images in, a grid out.

## Part E — Calibrate before you trust

*Reused in: every FID this course reports.*

A metric is not trusted because it is implemented. It is trusted because it was checked
against cases whose answer is known. **These three numbers go in your session log.**

1. **Split test** — MNIST cut in half at random; the two halves are independent draws
   from one distribution, so the true FID is **exactly zero**. Everything measured is
   bias.
2. **Sample-count curve** — the same split at several $n$. The bias falls like $1/n$.
3. **Garbage test** — noise images against MNIST. Enormous, and that range is the point.

Reference values from the course's own run (seed 0, pinned extractor, $n = 10^4$):
split test **0.603**, fitted log-log slope **−1.10**, garbage test **7028.6** — a factor
of about eleven thousand.

In [ ]:
from common.eval import MNISTFeatures, fid, ref_stats

extractor = MNISTFeatures(device=DEVICE)

# TODO: build the two halves of the MNIST train split, compute ref_stats on
#       half A, and score half B against it at n in {500, 2000, 10000}.
#       Assert the values are strictly positive and decreasing in n.
...

# TODO: the garbage test — uniform noise images against MNIST test.
#       Assert it exceeds the split test by at least a factor of 100.
...

### The payoff — the eye test becomes a number

In U0.L3 you trained two VAEs, at $d_z = 2$ and $d_z = 16$, and the wider one *looked*
better. That is where the course has been until today.

Load both checkpoints, sample ten thousand images from each, and score them. **At the
same sample count, with the same extractor** — which is the only reason the two numbers
may be put beside each other.

Course reference values: **918.8** at $d_z = 2$ against **296.0** at $d_z = 16$, a
factor of 3.1.

**Do not read anything into the absolute size.** The scale of an FID is a property of
the extractor: these are 128 unnormalized units from a small MNIST classifier, and the
values are not comparable with any published MNIST FID, which use Inception features.
Within this protocol they order models correctly; outside it they mean nothing.

One preprocessing decision hides inside the measurement. The decoder outputs Bernoulli
**probabilities**, and the samples could be those or binary draws from them. The
reference side is grayscale MNIST, and the extractor's preprocessing is named
`01-float-28x28`. Thresholding would compare a binary set against a grayscale one — a
protocol mismatch on the reference side, invisible in the resulting number. So score the
probabilities.

## Part F — Freeze

Commit both artifacts and tag **`u0l4-freeze`**.

```
common/unet.py   ->  unet-skeleton
common/eval.py   ->  eval-harness
```

From here, a change to either interface is a **breaking change** and needs a dated note
in the repository's `CHANGELOG`. This is the first API-stability lesson of the course
and it is not an exercise: U3.L2 imports `UNet` and trains it, U3.L3 switches on the
class pathway, U1.L2 imports `bpd`, PS0 imports everything.

**Extend without breaking:** add keyword-only arguments with defaults, never change the
positional contract — the `hygiene-stack` lesson from U0.L2, applied to a second
artifact. `UNet` already carries three such extras, and every call written against the
frozen five-argument form still works.

### PS0 is assembled

| Component | Built in | Role |
|---|---|---|
| `vae-mnist-scratch` | U0.L3 | the base codebase |
| `unet-skeleton` | **here** | replaces the encoder |
| `hygiene-stack` | U0.L2 | the training discipline |
| `eval-harness` | **here** | the numbers and the panels |

Graded on **correctness and reproducibility** — config, seed, commit hash, tracking
database — not on sample quality. Formally assigned at the close of U0.L5.

**Next session (U0.T5):** numerical solvers for ordinary differential equations — the
last drawer of the toolbox, and the one the second half of this course lives in. The
instrument you calibrated today is what makes the solver trade-off measurable.